In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col,lit,expr,regexp_replace,when,concat,lower,upper
from pyspark.sql.types import IntegerType,DoubleType,StringType,DataType


In [0]:

spark = SparkSession.builder.appName("Spark test").getOrCreate()


In [0]:

data = [
    ("101", "Asha",    "South", "Electronics", "1", "45000"),
    ("102", "Ravi",    "North", "Grocery", "5",  "1200"),
    ("103", "Meena",   "South", "Grocery","3",   "900"),
    ("104", "Karthik", "East",  "Electronics", "2", "60000"),
    ("105", "Divya",   "North", "Clothing", "4",  "3200"),
    ("106", "Imran",   "East",  "Clothing", "2", "1800"),
    ("107", "Sneha",   "South", "Electronics","1", "30000"),
    ("108", "Arjun",   "North", "Electronics","1", "52000"),
    ("109", "Kavya",   "East",  "Grocery", "6",  "1500"),
    ("110", "Rahul",   "South", "Clothing", "3",  "2700"),
]
cols = ["order_id", "customer", "region", "category", "qty", "amount"]

In [0]:
shop = spark.createDataFrame(data,cols)
shop.show()
shop.printSchema()

+--------+--------+------+-----------+---+------+
|order_id|customer|region|   category|qty|amount|
+--------+--------+------+-----------+---+------+
|     101|    Asha| South|Electronics|  1| 45000|
|     102|    Ravi| North|    Grocery|  5|  1200|
|     103|   Meena| South|    Grocery|  3|   900|
|     104| Karthik|  East|Electronics|  2| 60000|
|     105|   Divya| North|   Clothing|  4|  3200|
|     106|   Imran|  East|   Clothing|  2|  1800|
|     107|   Sneha| South|Electronics|  1| 30000|
|     108|   Arjun| North|Electronics|  1| 52000|
|     109|   Kavya|  East|    Grocery|  6|  1500|
|     110|   Rahul| South|   Clothing|  3|  2700|
+--------+--------+------+-----------+---+------+

root
 |-- order_id: string (nullable = true)
 |-- customer: string (nullable = true)
 |-- region: string (nullable = true)
 |-- category: string (nullable = true)
 |-- qty: string (nullable = true)
 |-- amount: string (nullable = true)



In [0]:

 shop.select("customer","category","region").display()

customer,category,region
Asha,Electronics,South
Ravi,Grocery,North
Meena,Grocery,South
Karthik,Electronics,East
Divya,Clothing,North
Imran,Clothing,East
Sneha,Electronics,South
Arjun,Electronics,North
Kavya,Grocery,East
Rahul,Clothing,South


In [0]:
shop.filter(shop.order_id.between(104,108)).display()

order_id,customer,region,category,qty,amount
104,Karthik,East,Electronics,2,60000
105,Divya,North,Clothing,4,3200
106,Imran,East,Clothing,2,1800
107,Sneha,South,Electronics,1,30000
108,Arjun,North,Electronics,1,52000


In [0]:
shop = shop.withColumn('region',
                     when (col('order_id') == "105", None)
                     .when(col('order_id') == "108", None)
                     .otherwise(col("region")))
shop.display()

order_id,customer,region,category,qty,amount
101,Asha,South,Electronics,1,45000
102,Ravi,North,Grocery,5,1200
103,Meena,South,Grocery,3,900
104,Karthik,East,Electronics,2,60000
105,Divya,null,Clothing,4,3200
106,Imran,East,Clothing,2,1800
107,Sneha,South,Electronics,1,30000
108,Arjun,null,Electronics,1,52000
109,Kavya,East,Grocery,6,1500
110,Rahul,South,Clothing,3,2700


In [0]:
shop = shop.withColumn('amount',
                     when (col('order_id') == "103", None)
                     .when(col('order_id') == "107", None)
                     .otherwise(col("amount")))
shop.display()

order_id,customer,region,category,qty,amount
101,Asha,South,Electronics,1,45000
102,Ravi,North,Grocery,5,1200
103,Meena,South,Grocery,3,null
104,Karthik,East,Electronics,2,60000
105,Divya,null,Clothing,4,3200
106,Imran,East,Clothing,2,1800
107,Sneha,South,Electronics,1,null
108,Arjun,null,Electronics,1,52000
109,Kavya,East,Grocery,6,1500
110,Rahul,South,Clothing,3,2700


In [0]:
shop = (shop.withColumn("amount", col("amount").cast("double"))
            .withColumn("order_id",col("order_id").cast("int"))
            .withColumn("qty",col("qty").cast("int")))
                       

In [0]:
shop.display()
shop.printSchema()

order_id,customer,region,category,qty,amount
101,Asha,South,Electronics,1,45000.0
102,Ravi,North,Grocery,5,1200.0
103,Meena,South,Grocery,3,null
104,Karthik,East,Electronics,2,60000.0
105,Divya,null,Clothing,4,3200.0
106,Imran,East,Clothing,2,1800.0
107,Sneha,South,Electronics,1,null
108,Arjun,null,Electronics,1,52000.0
109,Kavya,East,Grocery,6,1500.0
110,Rahul,South,Clothing,3,2700.0


root
 |-- order_id: integer (nullable = true)
 |-- customer: string (nullable = true)
 |-- region: string (nullable = true)
 |-- category: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- amount: double (nullable = true)



In [0]:
shop = shop.withColumn("Level",when(col("amount") >= 50000,"High").when(col("amount") >= 1000,"Mid").when(col("amount") < 1000,"Low"))

In [0]:
shop.display()

order_id,customer,region,category,qty,amount,Level
101,Asha,South,Electronics,1,45000.0,Mid
102,Ravi,North,Grocery,5,1200.0,Mid
103,Meena,South,Grocery,3,null,null
104,Karthik,East,Electronics,2,60000.0,High
105,Divya,null,Clothing,4,3200.0,Mid
106,Imran,East,Clothing,2,1800.0,Mid
107,Sneha,South,Electronics,1,null,null
108,Arjun,null,Electronics,1,52000.0,High
109,Kavya,East,Grocery,6,1500.0,Mid
110,Rahul,South,Clothing,3,2700.0,Mid


In [0]:
shop = shop.withColumn("Country",lit("India"))
shop.display()

order_id,customer,region,category,qty,amount,Level,Country
101,Asha,South,Electronics,1,45000.0,Mid,India
102,Ravi,North,Grocery,5,1200.0,Mid,India
103,Meena,South,Grocery,3,null,null,India
104,Karthik,East,Electronics,2,60000.0,High,India
105,Divya,null,Clothing,4,3200.0,Mid,India
106,Imran,East,Clothing,2,1800.0,Mid,India
107,Sneha,South,Electronics,1,null,null,India
108,Arjun,null,Electronics,1,52000.0,High,India
109,Kavya,East,Grocery,6,1500.0,Mid,India
110,Rahul,South,Clothing,3,2700.0,Mid,India


In [0]:
shop = shop.withColumn("Email_id",concat(lower(col("customer")),lit("123@gmail.com")))

In [0]:
shop.display()

order_id,customer,region,category,qty,amount,Level,Country,Email_id
101,Asha,South,Electronics,1,45000.0,Mid,India,asha123@gmail.com
102,Ravi,North,Grocery,5,1200.0,Mid,India,ravi123@gmail.com
103,Meena,South,Grocery,3,null,null,India,meena123@gmail.com
104,Karthik,East,Electronics,2,60000.0,High,India,karthik123@gmail.com
105,Divya,null,Clothing,4,3200.0,Mid,India,divya123@gmail.com
106,Imran,East,Clothing,2,1800.0,Mid,India,imran123@gmail.com
107,Sneha,South,Electronics,1,null,null,India,sneha123@gmail.com
108,Arjun,null,Electronics,1,52000.0,High,India,arjun123@gmail.com
109,Kavya,East,Grocery,6,1500.0,Mid,India,kavya123@gmail.com
110,Rahul,South,Clothing,3,2700.0,Mid,India,rahul123@gmail.com


In [0]:
shop = shop.withColumnsRenamed({
    "order_id": "ORDER_ID",
    "customer": "CUSTOMER",
    "region": "REGION",
    "category": "CATEGORY",
    "qty": "QTY",
    "amount": "AMOUNT",
    "country": "COUNTRY",
    "email_id": "EMAIL_ID"
})

In [0]:
shop.display()

ORDER_ID,CUSTOMER,REGION,CATEGORY,QTY,AMOUNT,Level,COUNTRY,EMAIL_ID
101,Asha,South,Electronics,1,45000.0,Mid,India,asha123@gmail.com
102,Ravi,North,Grocery,5,1200.0,Mid,India,ravi123@gmail.com
103,Meena,South,Grocery,3,null,null,India,meena123@gmail.com
104,Karthik,East,Electronics,2,60000.0,High,India,karthik123@gmail.com
105,Divya,null,Clothing,4,3200.0,Mid,India,divya123@gmail.com
106,Imran,East,Clothing,2,1800.0,Mid,India,imran123@gmail.com
107,Sneha,South,Electronics,1,null,null,India,sneha123@gmail.com
108,Arjun,null,Electronics,1,52000.0,High,India,arjun123@gmail.com
109,Kavya,East,Grocery,6,1500.0,Mid,India,kavya123@gmail.com
110,Rahul,South,Clothing,3,2700.0,Mid,India,rahul123@gmail.com


In [0]:
shop.withColumn('masked',regexp_replace(col('EMAIL_ID'),'[123]','*')).display()

ORDER_ID,CUSTOMER,REGION,CATEGORY,QTY,AMOUNT,Level,COUNTRY,EMAIL_ID,masked
101,Asha,South,Electronics,1,45000.0,Mid,India,asha123@gmail.com,asha***@gmail.com
102,Ravi,North,Grocery,5,1200.0,Mid,India,ravi123@gmail.com,ravi***@gmail.com
103,Meena,South,Grocery,3,null,null,India,meena123@gmail.com,meena***@gmail.com
104,Karthik,East,Electronics,2,60000.0,High,India,karthik123@gmail.com,karthik***@gmail.com
105,Divya,null,Clothing,4,3200.0,Mid,India,divya123@gmail.com,divya***@gmail.com
106,Imran,East,Clothing,2,1800.0,Mid,India,imran123@gmail.com,imran***@gmail.com
107,Sneha,South,Electronics,1,null,null,India,sneha123@gmail.com,sneha***@gmail.com
108,Arjun,null,Electronics,1,52000.0,High,India,arjun123@gmail.com,arjun***@gmail.com
109,Kavya,East,Grocery,6,1500.0,Mid,India,kavya123@gmail.com,kavya***@gmail.com
110,Rahul,South,Clothing,3,2700.0,Mid,India,rahul123@gmail.com,rahul***@gmail.com
